# Importing Libraries

In [10]:
import time
import chromadb

# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime


# Loading Documents

In [11]:
pdf_folder_location = "data/raw"

In [12]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [13]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

# Spliting into Chunks

In [14]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [15]:
amazon_chunks = pdf_loader.load_and_split(text_splitter)

In [16]:
len(amazon_chunks)

198

In [18]:
print(amazon_chunks[0])

page_content='ANNUAL REPORT
2 0 2 5' metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2022-02-14T21:08:55-06:00', 'author': 'T&C Composition', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc', 'trapped': '/False', 'source': 'data\\raw\\Amazon-2025-Annual-Report.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1'}


# Generating Embeddings

In [ ]:
amazon_collection = 'amazon_data'

In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [21]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Vikash Kumar\AppData\Local\Temp\ipykernel_11708\4093584008.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3132.63it/s]


In [22]:
chromadb_client = chromadb.PersistentClient(
    path=".data/processed"
)

In [23]:
chromadb_client.heartbeat()

1781242223946154000

In [24]:
chromadb_client.count_collections()

0

In [25]:
vectorstore = Chroma(
    collection_name=amazon_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory=".data/processed"
)

In [26]:
chromadb_client.count_collections()

1

In [27]:
chromadb_client.list_collections()

[Collection(name=amazon_data)]

In [ ]:
i = 0 # Initialize the starting index for the chunks

while i < len(amazon_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=amazon_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 5 seconds to avoid rate limiting issues with the vector store

In [29]:
vectorstore_persisted = Chroma(
    collection_name=amazon_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory=".data/processed"
)

In [30]:
collection = chromadb_client.get_collection(amazon_collection)

In [31]:
# Count the number of records in the collection
collection.count()

198

In [32]:
# Inspect the first 10 records
collection.peek()

{'ids': ['text_0',
  'text_1',
  'text_2',
  'text_3',
  'text_4',
  'text_5',
  'text_6',
  'text_7',
  'text_8',
  'text_9'],
 'embeddings': array([[-0.0246406 ,  0.04807853, -0.00068172, ...,  0.00178862,
          0.00688633, -0.0418699 ],
        [ 0.04931476,  0.04394592, -0.0198242 , ..., -0.01284927,
          0.03883617, -0.05404016],
        [ 0.03272779,  0.04328873, -0.04313071, ..., -0.02691266,
         -0.00378933, -0.05778611],
        ...,
        [ 0.03513523,  0.07104962, -0.01839117, ..., -0.00787499,
         -0.04287858, -0.04614053],
        [-0.00578766,  0.02619328, -0.02834646, ..., -0.0148871 ,
         -0.00942021, -0.01856914],
        [ 0.00477452,  0.06154822, -0.04631541, ...,  0.00387678,
         -0.03861846, -0.04757071]], shape=(10, 768)),
 'documents': ['ANNUAL REPORT\n2 0 2 5',
  'Dear Shareholders:\nWhen I graduated from college, I wanted to be a sportscaster. After sending my resume reel to many small\nmarkets around the U.S., and only getting tw

In [33]:
# tables / keys present in the vector DB
collection.peek().keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [34]:
# Inspect a specific record

collection.get(
    ids=['text_9']
)

{'ids': ['text_9'],
 'embeddings': None,
 'documents': ['1/ We have never seen a technology more quickly adopted than AI. When ChatGPT launched in November 2022,\nit reached 100 million users in two months—four times faster than TikT ok and 15 times faster than\nInstagram (ChatGPT already has over 900 million weekly active users). Both OpenAI and Anthropic have\nrevenue run rates reportedly approaching $30 billion. These are breathtaking numbers for companies this\nsoon after their commercial launches. When Edison opened his first commercial power station in 1882,\nmost people understood it as a better way to light a room. What they couldn’t see was that electricity would\neventually reorganize every factory, home, and industry on Earth. AI may have comparable impact. The\ndifference is that electricity took 40 years to get where it was going. AI appears to be moving ten times faster.\n2/ Amazon is smack in the middle of this land rush, and companies are choosing AWS for AI. Three year

# Retrieving Relevant Chunks

In [35]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [48]:
user_query = " What are Tesla’s main business segments described in the filings?"

In [39]:
relevant_document_chunks = retriever.invoke(user_query)

In [38]:
len(retriever.invoke(user_query))

5

In [40]:
for document in relevant_document_chunks:
    print(document)
    break


page_content='quality issues. In addition, profitability or other intended benefits, if any, in our newer activities (including development and 
adoption of automation, artificial intelligence, and machine learning technologies for customer and internal use), may not meet 
our expectations, and we may not be successful enough in these newer activities to recoup our investments in them, which 
investments are often significant. Failure to realize the benefits of amounts we invest in new technologies, products, or services 
could result in the value of those investments being written down or written off. In addition, our sustainability initiatives may be 
unsuccessful for a variety of reasons, including if we are unable to realize the expected benefits of new technologies or if we do 
not successfully plan or execute new strategies, which could harm our business or damage our reputation.
Our International Operations Expose Us to a Number of Risks
Our international activities are signific

In [41]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
quality issues. In addition, profitability or other intended benefits, if any, in our newer activities (including development and 
adoption of automation, artificial intelligence, and machine learning technologies for customer and internal use), may not meet 
our expectations, and we may not be successful enough in these newer activities to recoup our investments in them, which 
investments are often significant. Failure to realize the benefits of amounts we invest in new technologies, products, or services 
could result in the value of those investments being written down or written off. In addition, our sustainability initiatives may be 
unsuccessful for a variety of reasons, including if we are unable to realize the expected benefits of new technologies or if we do 
not successfully plan or execute new strategies, which could harm our business or damage our reputation.
Our International Operations Expose Us to a Number of Risks
Our international activities are 

# Creating the system prompt and retriving the relevant answers using RAG

In [52]:
qna_system_message = """
You are an enterprise knowledge assistant.
Answer the user question using only the provided context.
Rules:- Do not use outside knowledge.- If the answer is not available in the context, say: "I could not 
find this in the provided documents."- Cite the source file and page number or chunk ID for each key claim.- Do not invent numbers, dates, risks, or business conclusions.- Keep the answer clear and business-friendly.

Return:
1. User Query and then Direct Answer
2. Supporting Evidence
3. Sources Citations
4. Confidence: High / Medium / Low
5. Do not use external knowledge.
6. Do not hallucinate.
7. Do not invent facts.
8. If information is unavailable say:
   "I could not find this in the provided documents."
9. Show retrieved context snippets

User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [43]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [44]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [45]:
from groq import Groq
client = Groq()

In [ ]:
# model_name = 'openai/gpt-oss-120b'

In [53]:
user_query = " What risks are mentioned related to manufacturing or supply chain?"

In [51]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

**Direct Answer**  
The document identifies several manufacturing and supply‑chain risks, including:

1. **Limited or single‑source suppliers** – reliance on a small group of suppliers for key components (e.g., semiconductors/AI GPUs) can create shortages that hurt product development and service delivery.  
2. **Supplier disruptions** – bankruptcies, natural or human‑caused disasters, geopolitical events, labor and trade disputes, or other interruptions can prevent timely or affordable procurement of merchandise, content, components, or services.  
3. **Supplier non‑compliance** – violations of laws, regulations, intellectual‑property rights, or ethical standards by suppliers can lead to legal claims, reputational damage, and growth limits.  
4. **Inventory‑related manufacturing risks** – over‑ or under‑stocking, defective merchandise, spoilage, long lead‑times, pre‑payment requirements, and non‑returnable components can adversely affect operating results.  
5. **Logistics and shippin

In [54]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

**User Query:** What risks are mentioned related to manufacturing or supply chain?

**Direct Answer**  
The documents identify several manufacturing and supply‑chain risks, including:

1. **Limited or single‑source suppliers** – reliance on a small group of suppliers (e.g., for semiconductors) can create constraints on product availability and affect AI‑related development.  
2. **Supplier disruptions** – supplier bankruptcies, natural or human‑caused disasters, geopolitical events, labor and trade disputes, or decisions to limit/stop sales can prevent timely procurement of merchandise, components, or services.  
3. **Supplier compliance violations** – breaches of laws, regulations, intellectual‑property rights, or ethical standards by suppliers can lead to claims, reputational damage, and operational setbacks.  
4. **Inventory risk** – over‑stocking or under‑stocking due to seasonality, new product launches, rapid product‑cycle changes, defective goods, demand shifts, spoilage, long l

# Production Ready Function

In [55]:
import os
import chromadb

from groq import Groq

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

model_name = 'openai/gpt-oss-120b'
amazon_collection = 'amazon_data'

client = Groq()

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

chromadb_client = chromadb.PersistentClient(
    path=".data/processed"
)

vectorstore_persisted = Chroma(
    collection_name=amazon_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory=".data/processed"
)

retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

qna_system_message = """
You are an enterprise knowledge assistant.
Answer the user question using only the provided context.
Rules:- Do not use outside knowledge.- If the answer is not available in the context, say: "I could not 
find this in the provided documents."- Cite the source file and page number or chunk ID for each key claim.- Do not invent numbers, dates, risks, or business conclusions.- Keep the answer clear and business-friendly.

Return:
1. User Query and then Direct Answer
2. Supporting Evidence
3. Sources Citations
4. Confidence: High / Medium / Low
5. Do not use external knowledge.
6. Do not hallucinate.
7. Do not invent facts.
8. If information is unavailable say:
   "I could not find this in the provided documents."
9. Show retrieved context snippets

User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

def respond(user_query):
    relevant_document_chunks = retriever.invoke(user_query)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = "\n---\n".join(context_list)

    prompt = [
        {'role': 'system', 'content': qna_system_message},
        {
            'role': 'user', 'content': qna_user_message_template.format(
             context=context_for_query,
             question=user_query)
        }
    ]

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=prompt,
            temperature=0
        )

        answer = response.choices[0].message.content.strip()
    except Exception as e:
        answer = f'Sorry, I encountered the following error: \n {e}'

    return answer

def main():
    """
    Runs the main interactive loop for the Q&A system.

    This function initializes the conversation history, continuously prompts
    the user for queries, processes the queries using the `respond` function,
    and displays the assistant's responses. It also maintains the
    conversation history for context.

    Args:
        None

    Returns:
        None
    """

    conversation_history = [
        {'role': 'system', 'content': 'You are a helpful assistant who answers queries on financial documents'}
    ]

    while True:
        user_query = input("User (type q to quit): ")

        if user_query == 'q':
            break

        answer = respond(user_query)

        conversation_history.append({'role': 'user', 'content': user_query})
        conversation_history.append({'role': 'assistant', 'content': answer})

        print(f"Assistant: {answer}")

        print("----- Below is the conversation history -----")
        print(conversation_history)

if __name__ == "__main__":
    main()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6365.77it/s]


Assistant: **User Query:**  
What risks are mentioned related to manufacturing or supply chain?

**Direct Answer:**  
The documents identify several manufacturing and supply‑chain risks, including:

1. **Limited or single‑source suppliers** – reliance on a small number of vendors for key components (e.g., semiconductors) can create shortages that hinder product development and AI services.  
2. **Supplier instability** – supplier bankruptcies, decisions to limit or stop sales/licensing, and disruptions from natural or human‑caused disasters, geopolitical events, or labor and trade disputes can prevent timely procurement of merchandise, content, components, or services.  
3. **Supplier compliance failures** – violations of laws, regulations, intellectual‑property rights, or ethical standards by suppliers can expose the company to claims, reputational damage, growth limits, and adverse operating results.  
4. **Inventory‑related risks** – seasonality, new product launches, rapid product‑